# Pelatihan Model YOLOv11 — Deteksi Melanoma & Melanocytic Nevus

**Peneliti:** D. Ihsan Maulana — NIM 20220040069  
**Program Studi:** Teknik Informatika — Universitas Nusa Putra, Sukabumi — 2026

---

| Komponen | Detail |
|----------|--------|
| **Sumber** | Skin Lesions Dataset (Omiotek, 2026) — DOI: 10.5281/zenodo.18702734 |
| **Platform** | Roboflow (augmentasi 3x + export YOLOv11) |
| **Kelas** | 0=MEL (Melanoma Invasive), 1=NV (Melanocytic Nevus) |
| **Total** | 2.200 gambar (Train:1800 / Valid:200 / Test:200) |

## Langkah Kaggle:
1. Settings → Accelerator: **GPU T4 x2**
2. Settings → Internet: **ON**
3. Run All → tunggu 3-5 jam
4. Download ZIP dari tab Output

> Model ini adalah alat bantu skrining awal, bukan pengganti diagnosis medis.


---
## BAB 1 — Instalasi & Download Dataset


In [ ]:
# ================================================================
# Sel 1.1 - Install Pustaka + Download Dataset dari Roboflow
# ================================================================
import subprocess, sys

packages = [
    'ultralytics', 'roboflow', 'onnx', 'onnxruntime',
    'pandas', 'matplotlib', 'Pillow', 'tqdm', 'PyYAML',
]
print('Menginstal pustaka...')
for pkg in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])
    print('  OK:', pkg)

print('\nMendownload dataset dari Roboflow...')
from roboflow import Roboflow

RF_API_KEY   = 'IsEldcl0lmlPyroMkDuJ'
RF_WORKSPACE = 'ihsan-waeaz'
RF_PROJECT   = 'object-detection-kq9jg'
RF_VERSION   = 1

rf      = Roboflow(api_key=RF_API_KEY)
project = rf.workspace(RF_WORKSPACE).project(RF_PROJECT)
version = project.version(RF_VERSION)
dataset = version.download('yolov11')

ROBOFLOW_DATASET_LOCATION = dataset.location
print('Dataset berhasil didownload ke:', ROBOFLOW_DATASET_LOCATION)


In [ ]:
# ================================================================
# Sel 1.2 - Verifikasi GPU
# ================================================================
import torch, platform, os
import ultralytics

print('=' * 55)
print('INFORMASI LINGKUNGAN')
print('=' * 55)
print('  Python      :', platform.python_version())
print('  PyTorch     :', torch.__version__)
print('  Ultralytics :', ultralytics.__version__)
print('  CUDA Avail  :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('  GPU         :', torch.cuda.get_device_name(0))
    vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print('  VRAM        : %.1f GB' % vram)
else:
    print('  PERINGATAN: GPU tidak aktif!')
    print('  Kaggle: Settings > Accelerator > GPU T4 x2')

DEVICE    = 'cuda' if torch.cuda.is_available() else 'cpu'
IS_KAGGLE = os.path.exists('/kaggle/input')
PLATFORM  = 'Kaggle' if IS_KAGGLE else 'Colab/Lokal'
print('  Platform    :', PLATFORM)
print('  Device      :', DEVICE.upper())
print('=' * 55)


In [ ]:
# ================================================================
# Sel 1.3 - Konfigurasi & FIX data.yaml
# ================================================================
# Struktur folder Roboflow YOLOv11:
# /kaggle/working/Object-Detection-1/
#   data.yaml  <- path salah (../train/images), harus difix
#   train/images/  <- gambar ada DI SINI
#   train/labels/
#   valid/images/
#   valid/labels/
#   test/images/
#   test/labels/

from pathlib import Path
import yaml

DATASET_DIR  = Path(ROBOFLOW_DATASET_LOCATION)
DATASET_YAML = DATASET_DIR / 'data.yaml'

# ── Fix data.yaml dengan path ABSOLUT ────────────────────────────
# Roboflow kadang simpan path relatif yang salah (../train/images)
# Kita rewrite dengan absolute path yang pasti benar

fixed_yaml = {
    'train': str(DATASET_DIR / 'train' / 'images'),
    'val'  : str(DATASET_DIR / 'valid' / 'images'),
    'test' : str(DATASET_DIR / 'test'  / 'images'),
    'nc'   : 2,
    'names': ['MEL', 'NV'],
}

with open(DATASET_YAML, 'w') as f:
    yaml.dump(fixed_yaml, f, allow_unicode=True, default_flow_style=False)

print('data.yaml sudah difix dengan path absolut:')
for k, v in fixed_yaml.items():
    print('  %s: %s' % (k, v))

# ── Konfigurasi Eksperimen ────────────────────────────────────────
BASE_DIR  = Path('.')
PLOTS_DIR = BASE_DIR / 'plots'
PLOTS_DIR.mkdir(exist_ok=True)

MODEL_ARCH    = 'yolo11n.pt'
CLASS_NAMES   = ['MEL', 'NV']
RANDOM_SEED   = 42
EPOCHS        = 100
BATCH_SIZE    = 16
IMG_SIZE      = 640
LEARNING_RATE = 0.01
WEIGHT_DECAY  = 0.0005
PATIENCE      = 20
WORKERS       = 2
TARGET_MAP50  = 0.65
TARGET_PREC   = 0.70
TARGET_REC    = 0.65
EXPERIMENT_NAME = 'yolov11_melanoma_seed%d' % RANDOM_SEED

# ── Verifikasi Folder Dataset ─────────────────────────────────────
print('\n' + '=' * 55)
print('VERIFIKASI FOLDER DATASET')
print('=' * 55)
all_ok = True
for split in ['train', 'valid', 'test']:
    img_dir = DATASET_DIR / split / 'images'
    lbl_dir = DATASET_DIR / split / 'labels'
    n_img = len([f for f in img_dir.iterdir()
                 if f.suffix.lower() in ['.jpg','.jpeg','.png']]) if img_dir.exists() else 0
    n_lbl = len(list(lbl_dir.glob('*.txt'))) if lbl_dir.exists() else 0
    status = 'OK' if n_img > 0 else 'KOSONG'
    print('  [%s] %s | images: %d | labels: %d' % (split, status, n_img, n_lbl))
    if n_img == 0:
        all_ok = False

print('=' * 55)
if all_ok:
    print('Dataset OK - siap training!')
else:
    print('Ada folder kosong! Cek apakah download selesai.')

print('\nKonfigurasi:')
print('  Model     :', MODEL_ARCH)
print('  Epochs    :', EPOCHS)
print('  Batch     :', BATCH_SIZE)
print('  Eksperimen:', EXPERIMENT_NAME)


---
## BAB 2 — Eksplorasi Dataset (EDA)


In [ ]:
# ================================================================
# Sel 2.1 - Distribusi Kelas
# ================================================================
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import Counter

def count_classes(split):
    lbl_dir = DATASET_DIR / split / 'labels'
    counter = Counter({'MEL': 0, 'NV': 0})
    if not lbl_dir.exists():
        return counter
    for lf in lbl_dir.glob('*.txt'):
        with open(lf) as f:
            for line in f:
                parts = line.strip().split()
                if parts:
                    cid = int(parts[0])
                    if cid == 0: counter['MEL'] += 1
                    elif cid == 1: counter['NV'] += 1
    return counter

dist_train = count_classes('train')
dist_val   = count_classes('valid')
dist_test  = count_classes('test')

print('=' * 50)
print('DISTRIBUSI KELAS PER SPLIT')
print('=' * 50)
for name, dist in [('Train', dist_train), ('Valid', dist_val), ('Test', dist_test)]:
    total = sum(dist.values())
    print('  [%s] Total: %d' % (name, total))
    for cls, count in dist.items():
        pct = count / total * 100 if total > 0 else 0
        bar = '|' * int(pct / 3)
        print('    %s: %5d (%.1f%%) %s' % (cls, count, pct, bar))
print('=' * 50)

# Imbalance check
mel = dist_train.get('MEL', 1)
nv  = dist_train.get('NV', 1)
if mel > 0 and nv > 0:
    ratio = max(mel,nv)/min(mel,nv)
    print('Imbalance ratio: %.2f:1' % ratio)

# Grafik
COLORS = {'MEL': '#ef4444', 'NV': '#22c55e'}
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
fig.suptitle('Distribusi Kelas | Skin Lesions (Omiotek 2026)\nD. Ihsan Maulana, Teknik Informatika, Univ. Nusa Putra 2026',
             fontsize=10, fontweight='bold')

for ax, (sn, dist) in zip(axes, [('Training', dist_train), ('Validation', dist_val), ('Test', dist_test)]):
    total = sum(dist.values())
    classes = list(dist.keys())
    counts  = list(dist.values())
    colors  = [COLORS.get(c, '#94a3b8') for c in classes]
    if total == 0:
        ax.set_title('%s (kosong)' % sn)
        continue
    bars = ax.bar(classes, counts, color=colors, width=0.5)
    for bar, count in zip(bars, counts):
        pct = count/total*100
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+2,
                '%d\n(%.0f%%)' % (count, pct), ha='center', fontsize=9)
    ax.set_title('%s (n=%d)' % (sn, total), fontweight='bold')
    ax.set_ylim(0, max(counts)*1.3)
    ax.spines[['top','right']].set_visible(False)
    ax.grid(axis='y', alpha=0.3, linestyle='--')

legend = [mpatches.Patch(color=c, label='%s (%s)' % (n, 'Melanoma' if n=='MEL' else 'Nevus')) for n,c in COLORS.items()]
fig.legend(handles=legend, loc='lower center', ncol=2, bbox_to_anchor=(0.5,-0.08))
plt.tight_layout()
plt.savefig(str(PLOTS_DIR / '01_class_distribution.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Grafik distribusi kelas tersimpan.')


In [ ]:
# ================================================================
# Sel 2.2 - Tampilkan Sampel Citra Dermoskopi
# ================================================================
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image
import random

random.seed(RANDOM_SEED)

def get_samples(split, class_id, n=4):
    lbl_dir = DATASET_DIR / split / 'labels'
    img_dir = DATASET_DIR / split / 'images'
    results = []
    if not lbl_dir.exists():
        return results
    lfs = list(lbl_dir.glob('*.txt'))
    random.shuffle(lfs)
    for lf in lfs:
        with open(lf) as f:
            cids = [int(l.split()[0]) for l in f if l.strip()]
        if class_id in cids:
            for ext in ['.jpg','.jpeg','.png']:
                ip = img_dir / (lf.stem + ext)
                if ip.exists():
                    results.append((ip, lf))
                    break
        if len(results) >= n:
            break
    return results[:n]

def draw_bbox(ax, img, label_file):
    w, h = img.size
    clr = {0: '#ef4444', 1: '#22c55e'}
    names = {0: 'MEL', 1: 'NV'}
    with open(label_file) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 5: continue
            cid = int(parts[0])
            if len(parts) == 5:
                cx, cy, bw, bh = float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4])
            else:
                coords = [float(x) for x in parts[1:]]
                xs=coords[0::2]; ys=coords[1::2]
                cx=(min(xs)+max(xs))/2; cy=(min(ys)+max(ys))/2
                bw=max(xs)-min(xs); bh=max(ys)-min(ys)
            x1=(cx-bw/2)*w; y1=(cy-bh/2)*h
            color=clr.get(cid,'white')
            rect=mpatches.FancyBboxPatch((x1,y1),bw*w,bh*h,
                    linewidth=2,edgecolor=color,facecolor='none',boxstyle='square,pad=0')
            ax.add_patch(rect)
            ax.text(x1, max(y1-4,0), names.get(cid,str(cid)), color='white', fontsize=8, fontweight='bold',
                    bbox=dict(boxstyle='round,pad=0.2',fc=color,alpha=0.85))

samples_mel = get_samples('train', 0, n=4)
samples_nv  = get_samples('train', 1, n=4)

fig, axes = plt.subplots(2, 4, figsize=(16, 9))
fig.suptitle('Sampel Citra Dermoskopi — Skin Lesions (Omiotek 2026)\nD. Ihsan Maulana, Teknik Informatika, Univ. Nusa Putra 2026',
             fontsize=11, fontweight='bold')

for row, (cls_name, samples, color) in enumerate([
    ('MEL — Melanoma Invasive (Ganas)', samples_mel, '#ef4444'),
    ('NV — Melanocytic Nevus (Jinak)',  samples_nv,  '#22c55e')
]):
    for col in range(4):
        ax = axes[row][col]
        if col < len(samples):
            ip, lf = samples[col]
            img = Image.open(ip).convert('RGB')
            ax.imshow(img)
            draw_bbox(ax, img, lf)
            ax.set_title(ip.stem[:20], fontsize=7, color='gray')
        else:
            ax.text(0.5,0.5,'—',ha='center',va='center',transform=ax.transAxes)
        ax.axis('off')
    axes[row][0].set_ylabel(cls_name, fontsize=10, fontweight='bold', color=color, labelpad=10)

plt.tight_layout()
plt.savefig(str(PLOTS_DIR / '02_sample_images.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Visualisasi sampel tersimpan.')


---
## BAB 3 — Pelatihan Model YOLOv11

| Parameter | Nilai |
|-----------|-------|
| epochs | 100 |
| batch | 16 |
| imgsz | 640 |
| lr0 | 0.01 |
| patience | 20 |

**Estimasi: 3–5 jam di GPU T4. Jangan tutup browser!**


In [ ]:
# ================================================================
# Sel 3.1 - TRAINING YOLOv11
# ================================================================
from ultralytics import YOLO
import torch

torch.manual_seed(RANDOM_SEED)

print('=' * 55)
print('MEMULAI TRAINING YOLOv11')
print('  Model   :', MODEL_ARCH)
print('  Epochs  :', EPOCHS)
print('  Batch   :', BATCH_SIZE)
print('  Device  :', DEVICE.upper())
print('  data    :', str(DATASET_YAML))
print('=' * 55)

model = YOLO(MODEL_ARCH)

results = model.train(
    data=str(DATASET_YAML),
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    lr0=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    patience=PATIENCE,
    workers=WORKERS,
    seed=RANDOM_SEED,
    device=0 if torch.cuda.is_available() else 'cpu',
    name=EXPERIMENT_NAME,
    project='runs/detect',
    exist_ok=True,
    flipud=0.5,
    fliplr=0.5,
    mosaic=0.8,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    plots=True,
    save=True,
    save_period=10,
    verbose=True,
)

BEST_MODEL_PATH = Path('runs/detect/%s/weights/best.pt' % EXPERIMENT_NAME)
print('\n' + '=' * 55)
print('TRAINING SELESAI!')
print('  best.pt:', str(BEST_MODEL_PATH))
print('  Ada    :', BEST_MODEL_PATH.exists())
print('=' * 55)


---
## BAB 4 — Evaluasi Model


In [ ]:
# ================================================================
# Sel 4.1 - Evaluasi Validation Set
# ================================================================
from ultralytics import YOLO
import torch

best_model = YOLO(str(BEST_MODEL_PATH))

metrics = best_model.val(
    data=str(DATASET_YAML),
    imgsz=IMG_SIZE, conf=0.25, iou=0.45,
    device=0 if torch.cuda.is_available() else 'cpu',
    verbose=True,
)

precision = metrics.box.p.mean()
recall    = metrics.box.r.mean()
map50     = metrics.box.map50
map5095   = metrics.box.map
f1_score  = 2*(precision*recall)/(precision+recall+1e-8)

print('\n' + '=' * 55)
print('EVALUASI - VALIDATION SET')
print('=' * 55)
print('  Precision  : %.4f  (target >= %.2f)' % (precision, TARGET_PREC))
print('  Recall     : %.4f  (target >= %.2f)' % (recall, TARGET_REC))
print('  F1-Score   : %.4f' % f1_score)
print('  mAP@0.5    : %.4f  (target >= %.2f)' % (map50, TARGET_MAP50))
print('  mAP@0.5:95 : %.4f' % map5095)
print('=' * 55)


In [ ]:
# ================================================================
# Sel 4.2 - Evaluasi TEST SET (untuk BAB 4 Skripsi!)
# ================================================================
test_metrics = best_model.val(
    data=str(DATASET_YAML),
    split='test',
    imgsz=IMG_SIZE, conf=0.25, iou=0.45,
    device=0 if torch.cuda.is_available() else 'cpu',
    verbose=True,
)

test_precision = test_metrics.box.p.mean()
test_recall    = test_metrics.box.r.mean()
test_map50     = test_metrics.box.map50
test_map5095   = test_metrics.box.map
test_f1        = 2*(test_precision*test_recall)/(test_precision+test_recall+1e-8)

print('\n' + '=' * 55)
print('EVALUASI FINAL - TEST SET')
print('(Nilai ini untuk BAB 4 Skripsi!)')
print('=' * 55)
print('  Precision  : %.4f' % test_precision)
print('  Recall     : %.4f' % test_recall)
print('  F1-Score   : %.4f' % test_f1)
print('  mAP@0.5    : %.4f' % test_map50)
print('  mAP@0.5:95 : %.4f' % test_map5095)
print('=' * 55)


In [ ]:
# ================================================================
# Sel 4.3 - Training Curves
# ================================================================
import pandas as pd
import matplotlib.pyplot as plt

results_csv = Path('runs/detect/%s/results.csv' % EXPERIMENT_NAME)

if results_csv.exists():
    df = pd.read_csv(results_csv)
    df.columns = df.columns.str.strip()

    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    fig.suptitle('Training Curves — YOLOv11 Melanoma\nD. Ihsan Maulana, Teknik Informatika, Univ. Nusa Putra 2026',
                 fontsize=11, fontweight='bold')

    plot_cfg = [
        ('train/box_loss',       'Train Box Loss',  '#3b82f6', axes[0][0]),
        ('val/box_loss',         'Val Box Loss',    '#ef4444', axes[0][1]),
        ('metrics/mAP50(B)',     'mAP@0.5',         '#22c55e', axes[0][2]),
        ('metrics/precision(B)', 'Precision',       '#a855f7', axes[1][0]),
        ('metrics/recall(B)',    'Recall',          '#f59e0b', axes[1][1]),
        ('metrics/mAP50-95(B)',  'mAP@0.5:0.95',   '#06b6d4', axes[1][2]),
    ]

    for col, title, color, ax in plot_cfg:
        if col in df.columns:
            ax.plot(df['epoch'], df[col], color=color, linewidth=2)
            ax.set_title(title, fontweight='bold')
            ax.set_xlabel('Epoch')
            ax.grid(alpha=0.3, linestyle='--')
            ax.spines[['top','right']].set_visible(False)
        else:
            ax.set_title(title + ' (N/A)')
            ax.text(0.5, 0.5, 'kolom tidak tersedia', ha='center', va='center', transform=ax.transAxes, color='gray')

    plt.tight_layout()
    plt.savefig(str(PLOTS_DIR / '03_training_curves.png'), dpi=150, bbox_inches='tight')
    plt.show()
    print('Training curves tersimpan.')
else:
    print('results.csv tidak ditemukan.')


---
## BAB 5 — Export ONNX & Download Semua Hasil


In [ ]:
# ================================================================
# Sel 5.1 - Export Model ke ONNX
# ================================================================
from ultralytics import YOLO
import shutil

print('Mengekspor model ke ONNX...')
export_model = YOLO(str(BEST_MODEL_PATH))

onnx_path = export_model.export(
    format='onnx',
    imgsz=IMG_SIZE,
    dynamic=False,
    simplify=True,
    opset=12,
    half=False,
)

onnx_path = Path(onnx_path)
print('ONNX tersimpan:', str(onnx_path))
print('Ukuran: %.1f MB' % (onnx_path.stat().st_size / (1024**2)))

OUTPUT_WEIGHTS = Path('output_weights')
OUTPUT_WEIGHTS.mkdir(exist_ok=True)
shutil.copy2(BEST_MODEL_PATH, OUTPUT_WEIGHTS / 'best.pt')
shutil.copy2(onnx_path, OUTPUT_WEIGHTS / 'best.onnx')

print('\nFile tersimpan di output_weights/')
print('  best.pt   - PyTorch model')
print('  best.onnx - untuk deployment web')


In [ ]:
# ================================================================
# Sel 5.2 - Ringkasan Akhir & Simpan JSON
# ================================================================
import json
from datetime import datetime

summary = {
    'peneliti'    : 'D. Ihsan Maulana - NIM 20220040069',
    'universitas' : 'Teknik Informatika, Universitas Nusa Putra, Sukabumi 2026',
    'tanggal'     : datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'dataset': {
        'nama'    : 'Skin Lesions Dataset for 2-Class Object Segmentation',
        'sumber'  : 'Zenodo - Omiotek (2026) | DOI: 10.5281/zenodo.18702734',
        'roboflow': {'workspace': RF_WORKSPACE, 'project': RF_PROJECT, 'version': RF_VERSION},
        'total'   : 2200,
        'kelas'   : {'0': 'MEL (Melanoma)', '1': 'NV (Melanocytic Nevus)'},
    },
    'model'       : {'arsitektur': MODEL_ARCH, 'epochs': EPOCHS, 'batch': BATCH_SIZE},
    'metrik_test' : {
        'precision': round(float(test_precision), 4),
        'recall'   : round(float(test_recall), 4),
        'f1_score' : round(float(test_f1), 4),
        'map50'    : round(float(test_map50), 4),
        'map5095'  : round(float(test_map5095), 4),
    },
}

with open('experiment_summary.json', 'w') as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print('\n' + '=' * 60)
print('EKSPERIMEN SELESAI!')
print(json.dumps(summary, indent=2, ensure_ascii=False))
print('=' * 60)


In [ ]:
# ================================================================
# Sel 5.3 - ZIP SEMUA HASIL & SIAP DOWNLOAD
# ================================================================
import zipfile
from pathlib import Path

ZIP_NAME = 'melanoma_yolov11_results.zip'

print('Membuat ZIP semua hasil training...')

# Folder/file yang dimasukkan ke ZIP
to_include = [
    Path('output_weights'),
    Path('runs/detect') / EXPERIMENT_NAME,
    Path('plots'),
    Path('experiment_summary.json'),
]

with zipfile.ZipFile(ZIP_NAME, 'w', zipfile.ZIP_DEFLATED) as zf:
    for item in to_include:
        if item.is_dir():
            for file in item.rglob('*'):
                if file.is_file():
                    zf.write(file, file)
                    print('  +', str(file))
        elif item.is_file():
            zf.write(item, item)
            print('  +', str(item))
        else:
            print('  - Tidak ada:', str(item))

zip_size = Path(ZIP_NAME).stat().st_size / (1024**2)
print('\n' + '=' * 55)
print('ZIP SELESAI!')
print('  File : %s' % ZIP_NAME)
print('  Size : %.1f MB' % zip_size)
print('=' * 55)
print('\nISI ZIP:')
print('  output_weights/best.pt   <- PyTorch model')
print('  output_weights/best.onnx <- untuk web deployment!')
print('  runs/detect/.../results.csv <- metrik training')
print('  plots/*.png              <- grafik EDA & curves')
print('  experiment_summary.json  <- ringkasan metrik skripsi')
print('\nCara download di Kaggle:')
print('  Tab Output (kiri) > %s > Download' % ZIP_NAME)
